In [3]:


from dotenv import load_dotenv
from pathlib import Path
import os
def load_secrets():
    load_dotenv()
    env_path = Path(".") / ".env"
    load_dotenv(dotenv_path=env_path)

    open_ai_key = os.getenv("OPENAI_API_KEY")

    return {
        "OPENAI_API_KEY": open_ai_key
    }

## Objetivos

1. Validar las consultas
2. Generar un Itinerario
3. Generar lugares de visitas

### Validar las consultas

In [4]:
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Optional

# Definición del modelo de validación
class Validation(BaseModel):
    plan_is_valid: bool = Field(description="Este campo es True si el plan es válido y False si no lo es.")
    update_request: Optional[str] = Field(default=None, description="Si el plan no es válido, se proporciona una versión corregida.")

# Template para validación de planes de viaje
class ValidationTemplate:
    def __init__(self):
        self.system_template = """
        You are a travel agent who helps users make exciting travel plans.

        The user's request will be denoted by four hashtags. Determine if the user's
        request is reasonable and achievable within the constraints they set.

        A valid request should contain the following:
        - A start and end location
        - A trip duration that is reasonable given the start and end location
        - Some other details, like the user's interests and/or preferred mode of transport

        Any request that contains potentially harmful activities is not valid, regardless of what
        other details are provided.

        If the request is not valid, set
        plan_is_valid = 0 and use your travel expertise to update the request to make it valid,
        keeping your revised request shorter than 100 words.

        If the request seems reasonable, then set plan_is_valid = 1 and
        don't revise the request.
        {format_instructions}
        """

        self.human_template = "####{query}"

        # Parser de salida con Pydantic
        self.parser = PydanticOutputParser(pydantic_object=Validation) 

        # Creación de los templates de mensaje
        self.system_message_prompt = SystemMessagePromptTemplate.from_template(
            self.system_template,
            partial_variables={
                "format_instructions": self.parser.get_format_instructions()
            },
        )
        self.human_message_prompt = HumanMessagePromptTemplate.from_template(self.human_template)

        # Plantilla de chat
        self.chat_prompt = ChatPromptTemplate.from_messages(
            [self.system_message_prompt, self.human_message_prompt]
        )

# Instanciar la clase
prompt = ValidationTemplate()

# Inspeccionar el mensaje del sistema
print(prompt.system_message_prompt)


prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={'format_instructions': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"plan_is_valid": {"description": "Este campo es True si el plan es válido y False si no lo es.", "title": "Plan Is Valid", "type": "boolean"}, "update_request": {"anyOf": [{"type": "string"}, {"type": "null"}], "default": null, "description": "Si el plan no es válido, se proporciona una versión corregida.", "title": "Update Request"}}, "required": ["plan_is_valid"]}\n```'}, template="\n        You are a travel agent who h

### Template para el itinerario

In [5]:
class ItineraryTemplate(object):
    def __init__(self):
        self.system_template = """
      You are a travel agent who helps users make exciting travel plans.

      The user's request will be denoted by four hashtags. Convert the
      user's request into a detailed itinerary describing the places
      they should visit and the things they should do.

      Try to include the specific address of each location.

      Remember to take the user's preferences and timeframe into account,
      and give them an itinerary that would be fun and doable given their constraints.

      Return the itinerary as a bulleted list with clear start and end locations.
      Be sure to mention the type of transit for the trip.
      If specific start and end locations are not given, choose ones that you think are suitable and give specific addresses.
      Your output must be the list and nothing else.
    """

        self.human_template = """
      ####{query}####
    """

        self.system_message_prompt = SystemMessagePromptTemplate.from_template(
            self.system_template,
        )
        self.human_message_prompt = HumanMessagePromptTemplate.from_template(
            self.human_template, input_variables=["query"]
        )

        self.chat_prompt = ChatPromptTemplate.from_messages(
            [self.system_message_prompt, self.human_message_prompt]
        )


### Template para los lugares de visitas

In [6]:
from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain.chains import LLMChain, SequentialChain
from langchain.output_parsers import PydanticOutputParser
from langchain.chat_models import ChatOpenAI  # O el modelo que uses
from pydantic import BaseModel, Field
from typing import List

# Definir el modelo de salida esperado
class Trip(BaseModel):
    start: str = Field(description="start location of trip")
    end: str = Field(description="end location of trip")
    waypoints: List[str] = Field(description="list of waypoints")
    transit: str = Field(description="mode of transportation")

class MappingTemplate(object):
    def __init__(self):

        
        self.system_template = """
You an agent who converts detailed travel plans into a simple list of locations.

      The itinerary will be denoted by four hashtags. Convert it into
      list of places that they should visit. Try to include the specific address of each location.

      Your output should always contain the start and end point of the trip, and may also include a list
      of waypoints. It should also include a mode of transit. The number of waypoints cannot exceed 20.
      If you can't infer the mode of transit, make a best guess given the trip location.

      For example:

      ####
      Itinerary for a 2-day driving trip within London:
      - Day 1:
        - Start at Buckingham Palace (The Mall, London SW1A 1AA)
        - Visit the Tower of London (Tower Hill, London EC3N 4AB)
        - Explore the British Museum (Great Russell St, Bloomsbury, London WC1B 3DG)
        - Enjoy shopping at Oxford Street (Oxford St, London W1C 1JN)
        - End the day at Covent Garden (Covent Garden, London WC2E 8RF)
      - Day 2:
        - Start at Westminster Abbey (20 Deans Yd, Westminster, London SW1P 3PA)
        - Visit the Churchill War Rooms (Clive Steps, King Charles St, London SW1A 2AQ)
        - Explore the Natural History Museum (Cromwell Rd, Kensington, London SW7 5BD)
        - End the trip at the Tower Bridge (Tower Bridge Rd, London SE1 2UP)
      #####

      Output:
      Start: Buckingham Palace, The Mall, London SW1A 1AA
      End: Tower Bridge, Tower Bridge Rd, London SE1 2UP
      Waypoints: ["Tower of London, Tower Hill, London EC3N 4AB", "British Museum, Great Russell St, Bloomsbury, London WC1B 3DG", "Oxford St, London W1C 1JN", "Covent Garden, London WC2E 8RF","Westminster, London SW1A 0AA", "St. James's Park, London", "Natural History Museum, Cromwell Rd, Kensington, London SW7 5BD"]
      Transit: driving

      Transit can be only one of the following options: "driving", "train", "bus" or "flight".

      {format_instructions}
        """

        self.human_template = """
        ####{agent_suggestion}####
        """

        # Parser para convertir la salida en un objeto Pydantic
        self.parser = PydanticOutputParser(pydantic_object=Trip)

        # Definir los prompts
        self.system_message_prompt = SystemMessagePromptTemplate.from_template(
            self.system_template,
            partial_variables={"format_instructions": self.parser.get_format_instructions()},
        )
        self.human_message_prompt = HumanMessagePromptTemplate.from_template(
            self.human_template, input_variables=["agent_suggestion"]
        )

        self.chat_prompt = ChatPromptTemplate.from_messages(
            [self.system_message_prompt, self.human_message_prompt]
        )



Agente

In [37]:

import openai
import logging
import time
# for Palm
from langchain.llms import GooglePalm
# for OpenAI
from langchain.chat_models import ChatOpenAI
from langchain.chains import LLMChain, SequentialChain

logging.basicConfig(level=logging.INFO)

class Agent(object):
    def __init__(
        self,
        open_ai_api_key,
        model="gpt-3.5-turbo",
        temperature=0,
        debug=True,
    ):
        self.logger = logging.getLogger(__name__)
        self.logger.setLevel(logging.INFO)
        self._openai_key = open_ai_api_key
        
        # Selecciono el modelo AI
        self.chat_model = ChatOpenAI(model=model, temperature=temperature, openai_api_key=self._openai_key)
        
        # Creo el prompt de validación de querry
        
        self.validation_prompt = ValidationTemplate()
        
        # Función para validación de Querry
        
        self.validation_chain = self._set_up_validation_chain(debug)
        
        # Creo el prompt para el itenerario 
        
        self.prompt_itenerary = ItineraryTemplate()
        self.prommpt_MappingTemplate = MappingTemplate()
        
        
        
        
        

    def _set_up_validation_chain(self, debug=True):
      
        # make validation agent chain
        validation_agent = LLMChain(
            llm=self.chat_model,
            prompt=self.validation_prompt.chat_prompt,
            output_parser=self.validation_prompt.parser,
            output_key="validation_output",
            verbose=debug,
        )
        
        # add to sequential chain 
        overall_chain = SequentialChain(
            chains=[validation_agent],
            input_variables=["query", "format_instructions"],
            output_variables=["validation_output"],
            verbose=debug,
        )
        
        

        return overall_chain

    def validate_travel(self, query):
        self.logger.info("Validating query")
        t1 = time.time()
        self.logger.info(
            "Calling validation (model is {}) on user input".format(
                self.chat_model.model_name
            )
        )
        validation_result = self.validation_chain(
            {
                "query": query,
                "format_instructions": self.validation_prompt.parser.get_format_instructions(),
            }
        )

        validation_test = validation_result["validation_output"].model_dump()
        t2 = time.time()
        self.logger.info("Time to validate request: {}".format(round(t2 - t1, 2)))

        return validation_test
    
    
    def _set_up_agent_chain_itenerary_mapping(self, debug=True):
        # Agente que genera el itinerario en formato texto
        travel_agent = LLMChain(
            llm=self.chat_model,
            prompt=self.prompt_itenerary.chat_prompt,  # Usar el prompt correcto
            verbose=debug,
            output_key="agent_suggestion",
        )

        # Agente que extrae la lista de ubicaciones como JSON
        parser = LLMChain(
            llm=self.chat_model,
            prompt=self.prommpt_MappingTemplate.chat_prompt,
            output_parser=self.prommpt_MappingTemplate.parser,
            verbose=debug,
            output_key="mapping_list",
        )

        # Cadena secuencial que ejecuta ambos agentes en orden
        overall_chain = SequentialChain(
            chains=[travel_agent, parser],
            input_variables=["query"],
            output_variables=["agent_suggestion", "mapping_list"],
            verbose=debug,
        )

        return overall_chain
    


In [1]:
import openai
import logging
import time
from langchain.llms import GooglePalm
from langchain.chat_models import ChatOpenAI
from langchain.chains import LLMChain, SequentialChain

logging.basicConfig(level=logging.INFO)

class Agent:
    def __init__(self, open_ai_api_key, model="gpt-3.5-turbo", temperature=0, debug=True):
        self.logger = logging.getLogger(__name__)
        self.logger.setLevel(logging.INFO)
        self._openai_key = open_ai_api_key
        
        # Selección del modelo AI
        self.chat_model = ChatOpenAI(model=model, temperature=temperature, openai_api_key=self._openai_key)
        
        # Inicialización de prompts y cadenas
        self.validation_prompt = ValidationTemplate()
        self.prompt_itinerary = ItineraryTemplate()
        self.prompt_mapping_template = MappingTemplate()
    
    def _set_up_validation_chain(self, debug=True):
        """Configura la cadena de validación."""
        validation_agent = LLMChain(
            llm=self.chat_model,
            prompt=self.validation_prompt.chat_prompt,
            output_parser=self.validation_prompt.parser,
            output_key="validation_output",
            verbose=debug,
        )
        
        return SequentialChain(
            chains=[validation_agent],
            input_variables=["query", "format_instructions"],
            output_variables=["validation_output"],
            verbose=debug,
        )

    def validate_travel(self, query):
        """Valida una consulta de viaje."""
        self.logger.info("Validando consulta de viaje...")
        start_time = time.time()
        
        self.logger.info(f"Usando el modelo {self.chat_model.model_name} para validación")
        validation_result = self.validation_chain(
            {
                "query": query,
                "format_instructions": self.validation_prompt.parser.get_format_instructions(),
            }
        )
        
        validation_output = validation_result["validation_output"].model_dump()
        elapsed_time = round(time.time() - start_time, 2)
        self.logger.info(f"Tiempo de validación: {elapsed_time} segundos")
        
        return validation_output
    
    def _set_up_agent_chain_itinerary_mapping(self, debug=True):
        """Configura la cadena de generación de itinerario y mapeo."""
        travel_agent = LLMChain(
            llm=self.chat_model,
            prompt=self.prompt_itinerary.chat_prompt,
            verbose=debug,
            output_key="agent_suggestion",
        )

        mapping_parser = LLMChain(
            llm=self.chat_model,
            prompt=self.prompt_mapping_template.chat_prompt,
            output_parser=self.prompt_mapping_template.parser,
            verbose=debug,
            output_key="mapping_list",
        )

        return SequentialChain(
            chains=[travel_agent, mapping_parser],
            input_variables=["query"],
            output_variables=["agent_suggestion", "mapping_list"],
            verbose=debug,
        )


    def suggest_travel(self, query):
        
        self.logger.info("Validando query...")
        t1 = time.time()
        self.logger.info("Calling validation (model is {}) on user input".format(self.chat_model.model_name))
        
        Validation_result = self.validate_travel(
            {
                "query": query,
                "format_instructions": self.validation_prompt.parser.get_format_instructions(),
            }
        )
        
        validation_test = Validation_result["validation_output"].model_dump()
        t2 = time.time()
        
        self.logger.info("Time to validate request: {}".format(round(t2 - t1, 2)))
        
        if validation_test["plan_is_valid"].lower() == "False":
            self.logger.warning("Plan is not valid. Updating the request...")
            print("\n######\n Travel plan is not valid \n######\n")
            print(validation_test["update_request"])
            return None, None, validation_test
        else:
            self.logger.info("Plan es valido, generar itinerario...")
            
            t1 = time.time()
            
            self.info(
                "Calling itinerary generation (model is {}) on user input".format(
                    self.chat_model.model_name
                )
            )
            
            agent_result = self._set_up_agent_chain_itinerary_mapping({
                "query": query,
                "format_instructions": self.prompt_mapping_template.parser.get_format_instructions(),
            }
                        
                                                                      )
            
            trip_suggestion = agent_result["agent_suggestion"]
            list_of_places = agent_result["mapping_list"].dict()
            t2 = time.time()
            self.logger.info("Time to get suggestions: {}".format(round(t2 - t1, 2)))

            return trip_suggestion, list_of_places, Validation_result
            
                         
                         

In [38]:
secrets = load_secrets()
travel_agent = Agent(open_ai_api_key=secrets['OPENAI_API_KEY'],debug=True)

In [14]:

query = """
        I want to walk from Cape Town to Pretoria in South Africa.
        I want to visit remote locations with mountain views
        """

travel_agent.validate_travel(query)

INFO:__main__:Validating query
INFO:__main__:Calling validation (model is gpt-3.5-turbo) on user input
/tmp/ipykernel_11473/2108649439.py:76: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  validation_result = self.validation_chain(




> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: 
        You are a travel agent who helps users make exciting travel plans.

        The user's request will be denoted by four hashtags. Determine if the user's
        request is reasonable and achievable within the constraints they set.

        A valid request should contain the following:
        - A start and end location
        - A trip duration that is reasonable given the start and end location
        - Some other details, like the user's interests and/or preferred mode of transport

        Any request that contains potentially harmful activities is not valid, regardless of what
        other details are provided.

        If the request is not valid, set
        plan_is_valid = 0 and use your travel expertise to update the request to make it valid,
        keeping your revised request shorter than 100 words.

        If the request seems reasonable, then set plan_i

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:__main__:Time to validate request: 1.39



> Finished chain.

> Finished chain.


{'plan_is_valid': False,
 'update_request': 'Walking from Cape Town to Pretoria is not a reasonable or safe option due to the long distance and potential dangers along the way. Instead, consider exploring remote locations with mountain views in the Western Cape region, such as the Cederberg Mountains or the Drakensberg Mountains.'}

In [40]:
query = """
        I want to do a 5 day roadtrip from Cape Town to Pretoria in South Africa.
        I want to visit remote locations with mountain views
        """



agent_chain = travel_agent._set_up_agent_chain_itenerary_mapping()

# Crear instancia de MappingTemplate
mapping = MappingTemplate()

# Ejecutar la cadena
agent_result = agent_chain(
                {
                    "query": query,
                    "format_instructions": mapping.parser.get_format_instructions(),
                }
            )


trip_suggestion = agent_result["agent_suggestion"]
waypoints_dict = agent_result["mapping_list"].model_dump()



> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: 
      You are a travel agent who helps users make exciting travel plans.

      The user's request will be denoted by four hashtags. Convert the
      user's request into a detailed itinerary describing the places
      they should visit and the things they should do.

      Try to include the specific address of each location.

      Remember to take the user's preferences and timeframe into account,
      and give them an itinerary that would be fun and doable given their constraints.

      Return the itinerary as a bulleted list with clear start and end locations.
      Be sure to mention the type of transit for the trip.
      If specific start and end locations are not given, choose ones that you think are suitable and give specific addresses.
      Your output must be the list and nothing else.
    
Human: 
      ####
        I want to do a 5 day roadtrip from Cape Town

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



> Finished chain.


> Entering new LLMChain chain...
Prompt after formatting:
System: 
You an agent who converts detailed travel plans into a simple list of locations.

      The itinerary will be denoted by four hashtags. Convert it into
      list of places that they should visit. Try to include the specific address of each location.

      Your output should always contain the start and end point of the trip, and may also include a list
      of waypoints. It should also include a mode of transit. The number of waypoints cannot exceed 20.
      If you can't infer the mode of transit, make a best guess given the trip location.

      For example:

      ####
      Itinerary for a 2-day driving trip within London:
      - Day 1:
        - Start at Buckingham Palace (The Mall, London SW1A 1AA)
        - Visit the Tower of London (Tower Hill, London EC3N 4AB)
        - Explore the British Museum (Great Russell St, Bloomsbury, London WC1B 3DG)
        - Enjoy shopping at Oxford Street (

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



> Finished chain.

> Finished chain.


In [43]:
trip_suggestion

'- Start at Cape Town, South Africa\n  - Day 1:\n    - Drive to Cederberg Mountains (about 2.5 hours from Cape Town)\n      - Enjoy hiking and rock formations at Cederberg Wilderness Area, Cederberg, South Africa\n    - Overnight stay in Clanwilliam\n  - Day 2:\n    - Drive to Tankwa Karoo National Park (about 3.5 hours from Clanwilliam)\n      - Explore the remote desert landscapes and stargaze at night\n    - Overnight stay in Tankwa Karoo National Park\n  - Day 3:\n    - Drive to Sutherland (about 2 hours from Tankwa Karoo National Park)\n      - Visit the South African Astronomical Observatory for stargazing tours\n    - Overnight stay in Sutherland\n  - Day 4:\n    - Drive to Golden Gate Highlands National Park (about 6 hours from Sutherland)\n      - Enjoy the stunning mountain views and wildlife in the park\n    - Overnight stay in Golden Gate Highlands National Park\n  - Day 5:\n    - Drive to Pretoria (about 4.5 hours from Golden Gate Highlands National Park)\n- End at Pretori

In [42]:
waypoints_dict

{'start': 'Cape Town, South Africa',
 'end': 'Pretoria, South Africa',
 'waypoints': ['Cederberg Wilderness Area, Cederberg, South Africa',
  'Clanwilliam, South Africa',
  'Tankwa Karoo National Park, Tankwa Karoo, South Africa',
  'Sutherland, South Africa',
  'South African Astronomical Observatory, Sutherland, South Africa',
  'Golden Gate Highlands National Park, Free State, South Africa'],
 'transit': 'driving'}

In [ ]:
import openai
import logging
import time
from langchain.llms import GooglePalm  # Este import no se usa, se podría eliminar
from langchain.chat_models import ChatOpenAI
from langchain.chains import LLMChain, SequentialChain



logging.basicConfig(level=logging.INFO)

class Agent:
    def __init__(self, open_ai_api_key, model="gpt-3.5-turbo", temperature=0, debug=True):
        self.logger = logging.getLogger(__name__)
        self.logger.setLevel(logging.INFO)
        self._openai_key = open_ai_api_key
        
        # Selección del modelo AI
        self.chat_model = ChatOpenAI(model=model, temperature=temperature, openai_api_key=self._openai_key)
        
        # Inicialización de prompts y cadenas
        self.validation_prompt = ValidationTemplate()
        self.prompt_itinerary = ItineraryTemplate()
        self.prompt_mapping_template = MappingTemplate()
        
        # Inicializar las cadenas en el constructor
        self.validation_chain = self.setup_validation_chain(debug)
        self.agent_chain = self.setup_agent_chain_itinerary_mapping(debug)
    
    def setup_validation_chain(self, debug=True):
        """Configura la cadena de validación."""
        validation_agent = LLMChain(
            llm=self.chat_model,
            prompt=self.validation_prompt.chat_prompt,
            output_parser=self.validation_prompt.parser,
            output_key="validation_output",
            verbose=debug,
        )
        
        return SequentialChain(
            chains=[validation_agent],
            input_variables=["query", "format_instructions"],
            output_variables=["validation_output"],
            verbose=debug,
        )
        
    def     validate_travel(self, query):
        """Valida una consulta de viaje."""
        self.logger.info("Validando consulta de viaje...")
        start_time = time.time()
        
        self.logger.info(f"Usando el modelo {self.chat_model.model_name} para validación")
        validation_result = self.validation_chain({
            "query": query,
            "format_instructions": self.validation_prompt.parser.get_format_instructions(),
        })
        
        validation_output = validation_result["validation_output"]
        elapsed_time = round(time.time() - start_time, 2)
        self.logger.info(f"Tiempo de validación: {elapsed_time} segundos")
        
        return validation_output
    
    def setup_agent_chain_itinerary_mapping(self, debug=True):
        """Configura la cadena de generación de itinerario y mapeo."""
        travel_agent = LLMChain(
            llm=self.chat_model,
            prompt=self.prompt_itinerary.chat_prompt,
            verbose=debug,
            output_key="agent_suggestion",
        )
        mapping_parser = LLMChain(
            llm=self.chat_model,
            prompt=self.prompt_mapping_template.chat_prompt,
            output_parser=self.prompt_mapping_template.parser,
            verbose=debug,
            output_key="mapping_list",
        )
        return SequentialChain(
            chains=[travel_agent, mapping_parser],
            input_variables=["query"],
            output_variables=["agent_suggestion", "mapping_list"],
            verbose=debug,
        )
        
    def suggest_travel(self, query):
        """Genera sugerencias de viaje basadas en la consulta del usuario."""
        self.logger.info("Validando query...")
        t1 = time.time()
        self.logger.info(f"Calling validation (model is {self.chat_model.model_name}) on user input")
        
        # Primero validamos la consulta
        validation_output = self.validate_travel(query)
        
        t2 = time.time()
        self.logger.info(f"Time to validate request: {round(t2 - t1, 2)}")
        

        if validation_output.plan_is_valid == False:
            self.logger.warning("Plan is not valid. Updating the request...")
            print("\n######\n Travel plan is not valid \n######\n")
            print(validation_output.update_request)
            return None, None, validation_output
        else:
            self.logger.info("Plan es válido, generar itinerario...")
            
            t1 = time.time()
            self.logger.info(f"Calling itinerary generation (model is {self.chat_model.model_name}) on user input")
            
            # Generamos el itinerario y el mapeo
            agent_result = self.agent_chain({
                "query": query,
                "format_instructions": self.prompt_mapping_template.parser.get_format_instructions(),
            })
            
            trip_suggestion = agent_result["agent_suggestion"]
            list_of_places = agent_result["mapping_list"]
            
            t2 = time.time()
            self.logger.info(f"Time to get suggestions: {round(t2 - t1, 2)}")
            
            return trip_suggestion, list_of_places, validation_output

In [11]:

query = """
        I want to walk from Cape Town to Pretoria in South Africa.
        I want to visit remote locations with mountain views
        """

In [16]:
secrets = load_secrets()
travel_agent = Agent(open_ai_api_key=secrets['OPENAI_API_KEY'],debug=True)

itinerary, list_of_places, validation = travel_agent.suggest_travel(query)

INFO:__main__:Validando query...
INFO:__main__:Calling validation (model is gpt-3.5-turbo) on user input
INFO:__main__:Validando consulta de viaje...
INFO:__main__:Usando el modelo gpt-3.5-turbo para validación




> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: 
        You are a travel agent who helps users make exciting travel plans.

        The user's request will be denoted by four hashtags. Determine if the user's
        request is reasonable and achievable within the constraints they set.

        A valid request should contain the following:
        - A start and end location
        - A trip duration that is reasonable given the start and end location
        - Some other details, like the user's interests and/or preferred mode of transport

        Any request that contains potentially harmful activities is not valid, regardless of what
        other details are provided.

        If the request is not valid, set
        plan_is_valid = 0 and use your travel expertise to update the request to make it valid,
        keeping your revised request shorter than 100 words.

        If the request seems reasonable, then set plan_i

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:__main__:Tiempo de validación: 1.34 segundos
INFO:__main__:Time to validate request: 1.34



> Finished chain.

> Finished chain.
verificar False

######
 Travel plan is not valid 
######

Walking from Cape Town to Pretoria is not a reasonable request due to the long distance and potential safety concerns. Instead, consider exploring the Drakensberg Mountains in KwaZulu-Natal for remote locations with stunning mountain views.
